# Explore U.S. climbing route popularity
Use the main inner-joined dataset. Counts describe a historical user sample, not platform-wide totals. The archive was committed April 21, 2019; individual observation dates are unknown.

In [ ]:
from pathlib import Path
import pandas as pd
root = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
routes = pd.read_parquet(root / "data/processed/climbing_routes_main.parquet")
routes.shape

In [ ]:
routes[["route_name", "state", "grade", "protection_rating", "is_boulder", "sampled_climber_count", "sampled_tick_record_count"]].sample(15, random_state=593)

In [ ]:
routes.groupby("route_source").agg(routes=("mp_route_id", "size"), bouldering=("is_boulder", "sum"), median_sampled_climbers=("sampled_climber_count", "median"), ratings_available=("rating_record_count", "count"))

In [ ]:
routes.groupby("state").agg(routes=("mp_route_id", "size"), bouldering=("is_boulder", "sum"), median_sampled_climbers=("sampled_climber_count", "median")).sort_values("routes", ascending=False)

In [ ]:
coverage = pd.read_csv(root / "reports/main_join_coverage.csv")
coverage.groupby("state")[["input_routes", "retained_routes", "dropped_routes"]].sum().sort_values("retained_routes", ascending=False)

In [ ]:
routes.groupby(["grade_system", "grade"], dropna=False).agg(routes=("mp_route_id", "size"), median_sampled_climbers=("sampled_climber_count", "median")).sort_values("routes", ascending=False).head(30)

In [ ]:
routes.isna().mean().sort_values(ascending=False).head(20)

In [ ]:
supported = routes.loc[routes.has_at_least_5_sampled_climbers].copy()
supported.shape

## Interpretation
Report geographic and source coverage alongside correlations. Unmatched routes were dropped; absence does not mean zero activity. The archive caps user histories and has ambiguous repeated records, so distinct sampled climbers is the primary popularity measure. Ticks can include attempts. Keep YDS and V grades separate, preserve missing values, and compare results using the optional five-climber subset. Historical participation and newer features describe different snapshots.